# Mood-Based Music Recommendation System

In [ ]:
!pip -q install gradio opencv-python-headless tensorflow
print('Packages installed successfully.')

In [ ]:
!wget -q -O emotion_model.hdf5 https://github.com/oarriaga/face_classification/raw/master/trained_models/emotion_models/fer2013_mini_XCEPTION.102-0.66.hdf5
print('Pretrained CNN model downloaded successfully.')

In [ ]:
import cv2
import numpy as np
import gradio as gr
from urllib.parse import quote
from tensorflow.keras.models import load_model

emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

music_map = {
    'Happy': ['APT. - ROSÉ & Bruno Mars', 'Levitating - Dua Lipa', 'Ilahi - Arijit Singh', 'Uptown Funk - Bruno Mars'],
    'Sad': ['Channa Mereya - Arijit Singh', 'Agar Tum Saath Ho - Arijit Singh', 'The Night We Met - Lord Huron', 'Let Her Go - Passenger'],
    'Angry': ['Believer - Imagine Dragons', 'Numb - Linkin Park', 'Warriors - Imagine Dragons', 'Thunder - Imagine Dragons'],
    'Fear': ['Lovely - Billie Eilish & Khalid', 'Another Love - Tom Odell', 'Kun Faya Kun - A.R. Rahman', 'Phir Le Aya Dil - Arijit Singh'],
    'Disgust': ['Believer - Imagine Dragons', 'Numb - Linkin Park', 'Bones - Imagine Dragons', 'Centuries - Fall Out Boy'],
    'Surprise': ['On Top of the World - Imagine Dragons', "Can't Stop the Feeling! - Justin Timberlake", 'Shape of You - Ed Sheeran', 'Arabic Kuthu - Anirudh'],
    'Neutral': ['Until I Found You - Stephen Sanchez', 'Perfect - Ed Sheeran', 'Tum Se Hi - Mohit Chauhan', 'Samajavaragamana - Sid Sriram']
}

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
model = load_model('emotion_model.hdf5', compile=False)

print('Face detector and CNN model loaded successfully.')

In [ ]:
def detect_emotion(image):
    if image is None:
        return None, '## Please upload an image or use the webcam.'

    img = np.array(image).copy()
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40)
    )

    if len(faces) == 0:
        return img, '## No face detected. Use a clear front-facing image.'

    # Use the largest face if there are multiple faces.
    x, y, w, h = max(faces, key=lambda face: face[2] * face[3])
    face = gray[y:y+h, x:x+w]
    face = cv2.resize(face, (64, 64))
    face = face.astype('float32') / 255.0
    face = np.reshape(face, (1, 64, 64, 1))

    prediction = model.predict(face, verbose=0)[0]
    emotion_index = int(np.argmax(prediction))
    emotion = emotion_labels[emotion_index]
    confidence = float(prediction[emotion_index]) * 100

    cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.putText(
        img, f'{emotion}: {confidence:.1f}%', (x, max(y-10, 30)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
    )

    result = f'''# Detected Facial Expression: {emotion}

**Confidence: {confidence:.2f}%**

## Recommended Songs
'''

    for song in music_map[emotion]:
        spotify_url = f'https://open.spotify.com/search/{quote(song)}'
        result += f'\n- [{song}]({spotify_url})'

    if confidence < 55:
        result += '\n\n*Low confidence: please treat this result as a suggestion.*'

    return img, result

print('Emotion detection function created successfully.')

In [ ]:
with gr.Blocks(title='Mood-Based Music Recommendation') as demo:
    gr.Markdown('''# 🎵 Mood-Based Music Recommendation System

Upload a face image or use the webcam. The system uses a pretrained CNN to detect facial expression and recommends matching songs.''')

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(
                sources=['upload', 'webcam'], type='numpy', label='Upload Image / Webcam'
            )
            analyze_button = gr.Button('Detect Emotion and Recommend Songs', variant='primary')

        with gr.Column():
            output_image = gr.Image(label='Detection Result')
            output_text = gr.Markdown('Upload an image and click the button.')

    analyze_button.click(
        fn=detect_emotion,
        inputs=input_image,
        outputs=[output_image, output_text]
    )

    gr.Markdown('''**Technology used:** Python, OpenCV, TensorFlow/Keras, CNN, Gradio

*Facial expression is not a perfect measure of a person's actual mood.*''')

demo.launch(share=True, debug=True)